# SpaceX Launch Site Location Analysis


This notebook analyzes SpaceX Falcon 9 launch data to explore factors affecting first-stage landing success and build predictive models.


## Objective
- Map Falcon 9 launch sites and launch outcomes with Folium.
- Compare launch-site success markers geographically.
- Estimate distances from Cape Canaveral Launch Complex 40 to nearby landmarks.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / "data").exists() and (candidate / "notebooks").exists()
)
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOKS_DIR))

import pandas as pd

from notebook_utils import data_path, ensure_binary_file

try:
    import folium
    from folium.features import DivIcon
    from folium.plugins import MarkerCluster, MousePosition
except ImportError as exc:
    raise ImportError(
        "This notebook requires folium. Install the project dependencies before running it."
    ) from exc


## Data Loading


In [2]:
DATASET_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
)
dataset_path = data_path("spacex_launch_geo.csv")
if not dataset_path.exists():
    dataset_path = ensure_binary_file("spacex_launch_geo.csv", DATASET_URL)

spacex_df = pd.read_csv(dataset_path)
launch_sites_df = spacex_df[["Launch Site", "Lat", "Long"]].drop_duplicates().reset_index(drop=True)
launch_sites_df


,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,VAFB SLC-4E,34.632834,-120.610745
2,KSC LC-39A,28.573255,-80.646895
3,CCAFS SLC-40,28.563197,-80.576820


## Exploratory Data Analysis


In [3]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for _, row in launch_sites_df.iterrows():
    coordinate = [row["Lat"], row["Long"]]
    folium.Circle(
        coordinate,
        radius=1000,
        color="#264653",
        fill=True,
        fill_opacity=0.6,
    ).add_child(folium.Popup(row["Launch Site"])).add_to(site_map)
    folium.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 12px; color:#d35400;"><b>{row["Launch Site"]}</b></div>',
        ),
    ).add_to(site_map)

site_map


In [4]:
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

spacex_df = spacex_df.copy()
spacex_df["marker_color"] = spacex_df["class"].apply(lambda value: "green" if value == 1 else "red")

for _, row in spacex_df.iterrows():
    folium.Marker(
        [row["Lat"], row["Long"]],
        icon=folium.Icon(color="white", icon_color=row["marker_color"]),
    ).add_to(marker_cluster)

site_map


## Key Findings


In [5]:
from math import atan2, cos, radians, sin, sqrt


def calculate_distance(lat1, lon1, lat2, lon2):
    earth_radius_km = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return earth_radius_km * c


formatter = "function(num) {return L.Util.formatNum(num, 5);};"
site_map.add_child(
    MousePosition(
        position="topright",
        separator=" Long: ",
        empty_string="NaN",
        lng_first=False,
        num_digits=20,
        prefix="Lat:",
        lat_formatter=formatter,
        lng_formatter=formatter,
    )
)

launch_site_lat = 28.563197
launch_site_lon = -80.576820
landmarks = {
    "Coastline": (28.56334, -80.56799),
    "Highway": (28.56335, -80.57085),
    "Railroad": (28.57206, -80.58525),
    "City": (28.10473, -80.64531),
}

for _, coordinate in landmarks.items():
    distance = calculate_distance(launch_site_lat, launch_site_lon, coordinate[0], coordinate[1])
    folium.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 12px; color:#d35400;"><b>{distance:0.2f} km</b></div>',
        ),
    ).add_to(site_map)
    folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], coordinate], weight=1).add_to(site_map)

site_map
